<a href="https://colab.research.google.com/github/kat-ski/brachytherapy-dose/blob/main/Single_Seed_Dose_6711.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
#code written by Kat Skingle
#last date modified: 30.09.26

In [3]:
#libraries

import numpy as np
import os
import pandas as pd
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d

#connect Google Drive and link to project folder

from google.colab import drive
drive.mount('/content/drive')
project_folder = '/content/drive/MyDrive/Brachytherapy_Project'
#print(os.listdir(project_folder))

Mounted at /content/drive


In [4]:
#read the Excel sheet with values for the given seed
carleton_file = os.path.join(project_folder, 'I125_OncoSeed-6711.xlsx')
radial_dose = pd.read_excel(carleton_file, sheet_name = 'radial_dose', header=1, usecols="A:B", nrows = 36)
radial_dose.columns = ['r_cm', 'g_L'] #reading radial dose function at different radii away
#print(radial_dose)

"""
#plot the values of g_L against radial distance for fun!
plt.figure(figsize=(7, 4))
plt.plot(radial_dose['r_cm'], radial_dose['g_L'], 'o-')   # 'o-' = dots joined by lines

plt.axhline(1, color='grey', linestyle='--')   # dashed line at g_L = 1 for reference
plt.axvline(1, color='grey', linestyle='--')   # dashed line at r = 1 cm

plt.xlabel('Distance from seed, r (cm)')
plt.ylabel('Radial dose function, g_L(r)')
plt.title('Radial dose function for the 6711 seed (Carleton data)')
plt.show()
"""



"\n#plot the values of g_L against radial distance for fun!\nplt.figure(figsize=(7, 4))\nplt.plot(radial_dose['r_cm'], radial_dose['g_L'], 'o-')   # 'o-' = dots joined by lines\n\nplt.axhline(1, color='grey', linestyle='--')   # dashed line at g_L = 1 for reference\nplt.axvline(1, color='grey', linestyle='--')   # dashed line at r = 1 cm\n\nplt.xlabel('Distance from seed, r (cm)')\nplt.ylabel('Radial dose function, g_L(r)')\nplt.title('Radial dose function for the 6711 seed (Carleton data)')\nplt.show()\n"

In [5]:
#define g_L(r) for any radial distance, using a logarithmic function as it follows a roughly exponential decay relationship
#using Carleton values
# note that this is only betweeen 0.05cm and 10cm.


# Take the logarithm of g_L values
r_values = radial_dose['r_cm'].values
log_g_values = np.log(radial_dose['g_L'].values)

# Build an interpolation function in log space
# kind='linear' = straight lines between points (in log space)
log_g_interp = interp1d(r_values, log_g_values, kind='linear')

# Define a function that gives g_L at any distance r (in cm)
def g_L(r):
    # Interpolate the log, then undo the log with np.exp
    return np.exp(log_g_interp(r))

"""
# Tests that g_L is being read correctly
print(g_L(1.0))    # should be exactly 1
print(g_L(2.0))    # should match the table: about 0.8149
print(g_L(1.23))   # between the table values at 1.0 and 1.5
"""

'\n# Tests that g_L is being read correctly\nprint(g_L(1.0))    # should be exactly 1\nprint(g_L(2.0))    # should match the table: about 0.8149\nprint(g_L(1.23))   # between the table values at 1.0 and 1.5\n'

In [23]:
#dose-rate constant Λ for the 6711 seed, using cGy. Carleton value
dose_rate_constant = 0.932

In [11]:
#geometry function which describes how the dose spreads

L = 0.28   # active length of the seed, cm (Carleton)

def geometry_function(r, theta_deg):
    #Line-source geometry function G_L at distance r (cm) and angle theta (degrees)
    theta = np.radians(theta_deg)   # convert degrees to radians

    if np.isclose(np.sin(theta), 0): #at 0 degrees or 180 degrees in relation to the seed, x = 0
      return 1/(r**2 - ((L**2)/4))

    else:
      x = r*np.sin(theta)      # sideways distance from the seed's axis
      z = r*np.cos(theta)      # height along the seed's axis

      beta = np.arctan((z + L/2)/x) - np.arctan((z - L/2)/x)   # angle the line takes up, in radians

      return beta/(L*r*(np.sin(theta)))   # the formula for G_L

"""
print(geometry_function(1,90))
print(geometry_function(1,0))
print(geometry_function(1,180))
"""


0.9935424391576523
1.0199918400652794
1.0199918400652794


In [20]:
#anisotropy function which corrects for direction, based on the metal casing of the seed which blocks some radiation (dependent on direction and angle)
#reference point is when theta is 90 degrees (=1)

# header=2: use row 2 (the row of distances) as column names.
# nrows=32: read only the 32 angle rows (0 to 90 degrees), skipping the extras below
# usecols="A:M": the angle column plus the 12 distance columns.
anisotropy = pd.read_excel(carleton_file, sheet_name='Anisotropy',
                           header=2, nrows=32, usecols="A:M")

# Make the angle column the row labels, so rows are looked up by angle
anisotropy = anisotropy.set_index('Theta/deg')

# Replace the '-' placeholders with NaN (Python's "no value"), and make everything numbers
anisotropy = anisotropy.apply(pd.to_numeric, errors='coerce')

# Make sure the column labels (distances) are numbers too!
anisotropy.columns = anisotropy.columns.astype(float)

# print(anisotropy)

#interpolator function to find value at different radii

from scipy.interpolate import RegularGridInterpolator

# The grid: angles down the side, distances across the top, F values inside
angles = anisotropy.index.values.astype(float)    # 32 angles, 0 to 90 degrees
radii = anisotropy.columns.values                 # 12 distances, 0.1 to 10 cm
F_values = anisotropy.values                      # the 32 x 12 grid of F

# Build the 2D interpolator. The order (angles, radii) matches the grid's (rows, columns)
F_interp = RegularGridInterpolator((angles, radii), F_values, method='linear')

def anisotropy_function(r, theta_deg):
    #Anisotropy function F at distance r (cm) and angle theta (degrees), Carleton data
    if theta_deg >= 90:                     # the angle is more than 90 degrees
        theta_deg = 180 - theta_deg         # use the matching angle on the other side
    return F_interp([theta_deg, r])[0]


"""
#test
print(anisotropy_function(1,90))
print(anisotropy_function(1,0))
print(anisotropy_function(1, 180))
print(anisotropy_function(1.5,0))
"""



1.0
0.2887
0.2887
0.33805


In [24]:
#calculate dose rate, as seed strength x dose rate constant x geometry function/reference point value x radial dose function x anistropy function
# S_K as seed strength will be set as 1U in initial calculations, but typical strengths are 0.4-0.6U for prostate seeds
def dose_rate(r, theta_deg, S_K):
  return S_K * dose_rate_constant * (geometry_function(r, theta_deg)/geometry_function(1,90))* g_L(r)* anisotropy_function(r, theta_deg)

"""
#test
print(dose_rate(1, 90, 1))   # should be 0.932
print(dose_rate(2, 90, 1))
print(dose_rate(1, 0, 1))
"""


0.932
0.1907871537156454
0.276231353189204
